<a href="https://colab.research.google.com/github/kkm1121/ndvia-yolo/blob/20250404/ndvia_yolo_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import cv2
import torch
import yt_dlp  # 유튜브 영상 다운로드 라이브러리
from ultralytics import YOLO
from moviepy.editor import VideoFileClip
from IPython.display import display, Video

# 🔹 1️⃣ 유튜브에서 480p 영상 다운로드
def download_youtube_video(youtube_url, output_path):
    ydl_opts = {
        'format': 'best[height<=480]',  # 480p로 다운로드
        'outtmpl': output_path
    }
    with yt_dlp.YoutubeDL(ydl_opts) as ydl:
        ydl.download([youtube_url])
    print(f"✅ 유튜브 영상 다운로드 완료: {output_path}")

# 🔹 2️⃣ 특정 구간(30~35초)만 자르기
def trim_video(input_path, output_path, start_time=30, end_time=35):
    clip = VideoFileClip(input_path).subclip(start_time, end_time)
    clip.write_videofile(output_path, codec='libx264')
    print(f"✅ 영상 잘라내기 완료: {output_path}")

# 🔹 3️⃣ YOLOv8 자동차 감지 (중복 방지)
def detect_cars(video_path, output_path):
    model = YOLO("yolov8n.pt")  # YOLOv8 모델 로드
    cap = cv2.VideoCapture(video_path)

    # 비디오 저장 설정
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(cap.get(cv2.CAP_PROP_FPS))
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    out = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    detected_cars = set()  # 🚗 자동차 ID 저장 (중복 방지)
    frame_count = 0

    while cap.isOpened():
        ret, frame = cap.read()
        if not ret:
            break

        results = model(frame)  # YOLO 감지 실행
        frame_count += 1

        for r in results:
            for box in r.boxes:
                class_id = int(box.cls[0])
                if class_id in [2, 3, 5, 7]:  # 🚗 자동차(2), 트럭(3), 버스(5), 트레일러(7) 감지
                    x1, y1, x2, y2 = map(int, box.xyxy[0])  # 바운딩 박스 좌표
                    center_x = (x1 + x2) // 2  # 자동차 중심 좌표 계산
                    center_y = (y1 + y2) // 2
                    car_id = f"{center_x}-{center_y}"  # 🚗 자동차 고유 ID 생성

                    if car_id not in detected_cars:
                        detected_cars.add(car_id)  # 중복 방지용 저장
                        print(f"🚗 새로운 자동차 감지! (ID: {car_id})")

                    # 감지된 자동차 박스 그리기
                    cv2.rectangle(frame, (x1, y1), (x2, y2), (0, 255, 0), 3)
                    cv2.putText(frame, "Car", (x1, y1 - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 255, 0), 2)

        out.write(frame)  # 처리된 프레임 저장

    cap.release()
    out.release()

    print(f"🚗 감지된 자동차 총 개수: {len(detected_cars)}")
    print(f"✅ 자동차 감지 영상 저장 완료: {output_path}")

# 🔹 4️⃣ Colab에서 HTML5 플레이어로 영상 보기
def show_video(video_path):
    print(f"📽️ 영상 재생: {video_path}")
    display(Video(video_path, embed=True))

# 🚀 실행 순서
youtube_url = "https://www.youtube.com/watch?v=QtO6I9tGdbg"  # 원하는 유튜브 영상 링크
download_youtube_video(youtube_url, "original.mp4")  # 유튜브에서 다운로드
trim_video("original.mp4", "trimmed.mp4", start_time=30, end_time=35)  # 30~35초 구간 추출
detect_cars("trimmed.mp4", "car_detected.mp4")  # YOLO 감지 실행
show_video("car_detected.mp4")  # Colab에서 보기


[youtube] Extracting URL: https://www.youtube.com/watch?v=QtO6I9tGdbg
[youtube] QtO6I9tGdbg: Downloading webpage
[youtube] QtO6I9tGdbg: Downloading tv client config
[youtube] QtO6I9tGdbg: Downloading player 73381ccc-main
[youtube] QtO6I9tGdbg: Downloading tv player API JSON
[youtube] QtO6I9tGdbg: Downloading ios player API JSON
[youtube] QtO6I9tGdbg: Downloading m3u8 information
[info] QtO6I9tGdbg: Downloading 1 format(s): 18
[download] Destination: original.mp4
[download] 100% of    6.00MiB in 00:00:01 at 3.69MiB/s   
✅ 유튜브 영상 다운로드 완료: original.mp4
Moviepy - Building video trimmed.mp4.
MoviePy - Writing audio in trimmedTEMP_MPY_wvf_snd.mp3


MoviePy - Done.
Moviepy - Writing video trimmed.mp4



Moviepy - Done !
Moviepy - video ready trimmed.mp4
✅ 영상 잘라내기 완료: trimmed.mp4

0: 384x640 3 persons, 1 car, 1 sandwich, 1 donut, 770.6ms
Speed: 5.8ms preprocess, 770.6ms inference, 2.1ms postprocess per image at shape (1, 3, 384, 640)
🚗 새로운 자동차 감지! (ID: 153-240)

0: 384x640 3 persons, 1 sandwich, 1 donut, 399.2ms
Speed: 2.3ms preprocess, 399.2ms inference, 11.5ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 sandwich, 1 donut, 588.5ms
Speed: 2.3ms preprocess, 588.5ms inference, 1.7ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 2 persons, 1 donut, 417.6ms
Speed: 2.2ms preprocess, 417.6ms inference, 1.4ms postprocess per image at shape (1, 3, 384, 640)

0: 384x640 3 persons, 1 car, 1 sandwich, 1 donut, 398.6ms
Speed: 2.3ms preprocess, 398.6ms inference, 14.0ms postprocess per image at shape (1, 3, 384, 640)
🚗 새로운 자동차 감지! (ID: 462-216)

0: 384x640 2 persons, 2 cars, 1 truck, 1 sandwich, 1 donut, 237.1ms
Speed: 2.3ms preprocess, 237.1ms inference, 